<a href="https://colab.research.google.com/github/joaoguilherme-devsec/RVRT-Video-Restoration/blob/main/RVRT_demo_on_Video_Restoration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVRT Online Demo on Video Restoration

[![arXiv](https://img.shields.io/badge/arXiv-Paper-<COLOR>.svg)](https://arxiv.org/abs/2205.0000)
[![GitHub Stars](https://img.shields.io/github/stars/JingyunLiang/RVRT?style=social)](https://github.com/JingyunLiang/VRT)
[![download](https://img.shields.io/github/downloads/JingyunLiang/RVRT/total.svg)](https://github.com/JingyunLiang/VRT/releases)
![visitors](https://visitor-badge.glitch.me/badge?page_id=jingyunliang/RVRT)

This is a **RVRT online demo on Video Restoration** of our paper [''Recurrent Video Restoration Transformer with Guided Deformable Attention''](https://arxiv.org/abs/2205.00000).

VRT ahcieves state-of-the-art performance with balanced model size, testing memory, and runtime in
- video SR (REDS, Vimeo90K, Vid4 and UDM10)
- video deblurring (GoPro, DVD and REDS)
- video denoising (DAVIS and Set8)

<img src="https://raw.githubusercontent.com/JingyunLiang/RVRT/main/assets/vsr_visual.jpg" width="100%">
<img src="https://raw.githubusercontent.com/JingyunLiang/RVRT/main/assets/vdb_visual.jpg" width="100%">
<img src="https://raw.githubusercontent.com/JingyunLiang/RVRT/main/assets/vdn_visual.jpg" width="100%">



# 1. Preparations
Before start, make sure that you choose
* Runtime Type = Python 3
* Hardware Accelerator = GPU
* Broswer != Firefox (cannot upload images in step 2)

in the **Runtime** menu -> **Change runtime type**

Then, we clone the repository, set up the envrironment, and download the pre-trained model.

In [24]:
!rm -r RVRT
# Clone RVRT
!git clone https://github.com/JingyunLiang/RVRT.git
%cd RVRT
!pip install -r requirements.txt

rm: cannot remove 'RVRT': No such file or directory
Cloning into 'RVRT'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 48 (delta 3), reused 2 (delta 2), pack-reused 37 (from 1)
Receiving objects: 100% (48/48), 2.87 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/RVRT
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 2.6 MB/s eta 0:00:00


# 2. Upload Images

Upload the images to be processed by RVRT

In [25]:
import os
import glob
from google.colab import files
import shutil
print(' Note1: You can upload a single video/ video frames as a batch. \n Note2: If you just to want to replicate the results in the paper, you can skip this step (we have enabled automatic download in the code).\n Note3: You may need Chrome to enable file uploading!\n Note4: Our model is trained on simple synthetic videos (e.g., bicubic). It may not perform well on real-world videos.\n')
input_folder = None

# upload images
!rm -r testsets/*
!rm -r results/*
os.mkdir('testsets/uploaded')
os.mkdir('testsets/uploaded/000')
uploaded = files.upload()
for filename in uploaded.keys():
  is_image = (filename.split('.')[-1].lower() in ['png','jpeg','jpg','tiff','bmp','tif',])
  input_folder = 'testsets/uploaded/000' if is_image else 'testsets/uploaded'
  dst_path = os.path.join(input_folder, filename)
  print(f'move {filename} to {dst_path}')
  shutil.move(filename, dst_path)

  if not is_image: # image or video
    print(f'extracting the video as frames to {input_folder}/000')
    os.system(f'ffmpeg -i testsets/uploaded/{filename} -qscale:v 1 -qmin 1 -qmax 1 -vsync 0  {input_folder}/000/frame%08d.png')
    os.remove(f'testsets/uploaded/{filename}')

!ls testsets/uploaded/000

 Note1: You can upload a single video/ video frames as a batch. 
 Note2: If you just to want to replicate the results in the paper, you can skip this step (we have enabled automatic download in the code).
 Note3: You may need Chrome to enable file uploading!
 Note4: Our model is trained on simple synthetic videos (e.g., bicubic). It may not perform well on real-world videos.

rm: cannot remove 'results/*': No such file or directory


Saving CIMG0120.mp4 to CIMG0120.mp4
move CIMG0120.mp4 to testsets/uploaded/CIMG0120.mp4
extracting the video as frames to testsets/uploaded/000
frame00000001.png  frame00000297.png  frame00000593.png  frame00000889.png
frame00000002.png  frame00000298.png  frame00000594.png  frame00000890.png
frame00000003.png  frame00000299.png  frame00000595.png  frame00000891.png
frame00000004.png  frame00000300.png  frame00000596.png  frame00000892.png
frame00000005.png  frame00000301.png  frame00000597.png  frame00000893.png
frame00000006.png  frame00000302.png  frame00000598.png  frame00000894.png
frame00000007.png  frame00000303.png  frame00000599.png  frame00000895.png
frame00000008.png  frame00000304.png  frame00000600.png  frame00000896.png
frame00000009.png  frame00000305.png  frame00000601.png  frame00000897.png
frame00000010.png  frame00000306.png  frame00000602.png  frame00000898.png
frame00000011.png  frame00000307.png  frame00000603.png  frame00000899.png
frame00000012.png  frame0000030

# 3. Inference


In [ ]:
import os
import shutil
from pathlib import Path

# Caminhos das pastas
original_uploaded_dir = Path('testsets/uploaded/000')
temp_processing_dir = Path('testsets/processing_temp/000')

# Limpa qualquer resíduo anterior
if temp_processing_dir.exists():
    shutil.rmtree(temp_processing_dir.parent)

os.makedirs(temp_processing_dir)

# Seleciona uma janela estável de 30 frames para evitar o estouro de VRAM da GPU
all_frames = sorted(list(original_uploaded_dir.glob('*.png')))
selected_frames = all_frames[:30]

print(f'Copiando {len(selected_frames)} frames para processamento estável na GPU...')
for i, frame_path in enumerate(selected_frames):
    shutil.copy(frame_path, temp_processing_dir / f'frame_{i:08d}.png')

# Otimizações extras de memória para o PyTorch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:32'

print('\nIniciando a inferência estável...')
# Executa apontando para a pasta temporária menor
!python main_test_rvrt.py --task 001_RVRT_videosr_bi_REDS_30frames --folder_lq testsets/processing_temp --tile 0 64 64 --tile_overlap 2 20 20 --num_workers 1 --save_result

print('\nInferência concluída com sucesso!')

Copiando 30 frames para processamento estável na GPU...

Iniciando a inferência estável...


# 4. Visualization

In [ ]:
# utils for visualization
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

def display(img1, name='input'):
  num_frames = len(img1)
  if num_frames == 0:
    print(f'\n[Aviso] Nenhuma imagem encontrada para "{name}". Verifique os caminhos de origem.')
    return

  # Mostra até 5 frames dependendo de quantos estão disponíveis
  total_figs = min(5, num_frames)
  fig, axes = plt.subplots(1, total_figs, figsize=(total_figs * 12, 14))

  if total_figs == 1:
    axes = [axes]

  for i in range(total_figs):
    axes[i].set_title(f'Frame {i} ({name})', fontsize=30)
    axes[i].axis('off')
    axes[i].imshow(img1[i])
  plt.show()

# display video by frames
print(' Note: Due to Colab GPU memory limitation, we use smaller --tile here, which may lead to different performance from the paper. \n Please refer to https://github.com/JingyunLiang/RVRT#quick-testing for official arguments.\n')

# Busca os inputs de baixa resolução que foram realmente usados na pasta temporária
img_in = []
for path in sorted(Path('testsets/processing_temp').rglob('*.png'))[:10]:
  img_in.append(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
display(img_in, 'input')

# Busca os outputs de alta resolução restaurados pelo RVRT na pasta de resultados
img_out = []
for path in sorted(Path('results').rglob('*.png'))[:10]:
  img_out.append(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
display(img_out, 'RVRT output')

# 5. Download Results


In [ ]:
# Download the results
zip_filename = 'RVRT_result.zip'
!tar -czf {zip_filename} results/*
files.download(zip_filename)

```markdown
# 6. Processamento Completo de Máxima Qualidade (Todos os 1184 Frames)
Abaixo realizamos o processamento em lote de todos os frames utilizando o algoritmo em sua máxima capacidade (sem o parâmetro `--tile` limitador de qualidade).
```

In [ ]:
import os
import gc
import torch
import shutil
from pathlib import Path

# Caminhos absolutos mapeando os locais reais no Colab
path_option_1 = Path('/content/RVRT/testsets/uploaded/000')
path_option_2 = Path('/content/testsets/uploaded/000')

# Determina qual pasta de fato contém os frames extraídos
if path_option_1.exists() and len(list(path_option_1.glob('*.png'))) > 0:
    original_dir = path_option_1
else:
    original_dir = path_option_2

temp_parent = Path('/content/RVRT/testsets/processing_temp')
temp_sub = temp_parent / '000'
final_results_dir = Path('/content/RVRT/results/001_RVRT_videosr_bi_REDS_30frames/processing_temp')

# Obter e ordenar todos os frames originais
all_frames = sorted(list(original_dir.glob('*.png')))
total_frames = len(all_frames)

# O modelo RVRT REDS_30frames exige EXATAMENTE lotes múltiplos de 30 frames
chunk_size = 30

print(f'Diretório de origem detectado: {original_dir}')
print(f'Total de frames originais para processar: {total_frames}')
print('Processando com QUALIDADE MÁXIMA (sem sub-divisão de tiles, lotes de 30 frames)...')

# Garantir que as pastas de destino existam e estejam limpas antes do início
if final_results_dir.exists():
    shutil.rmtree(final_results_dir)

# Configurações de otimização de memória GPU
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'

# Execução em lote sequencial
if total_frames > 0:
    for start_idx in range(0, total_frames, chunk_size):
        end_idx = start_idx + chunk_size
        print(f'--> [Lote] Processando do frame {start_idx + 1} ao {min(end_idx, total_frames)}...')

        # Recriar pasta temporária limpa para o lote atual
        if temp_parent.exists():
            shutil.rmtree(temp_parent)
        os.makedirs(temp_sub)

        # Copiar os frames deste lote
        batch_frames = all_frames[start_idx:min(end_idx, total_frames)]
        while len(batch_frames) < chunk_size:
            batch_frames.append(batch_frames[-1])

        for i, frame_path in enumerate(batch_frames):
            shutil.copy(frame_path, temp_sub / f'frame_{start_idx + i:08d}.png')

        # Executa a inferência de máxima qualidade para o lote
        !python /content/RVRT/main_test_rvrt.py --task 001_RVRT_videosr_bi_REDS_30frames --folder_lq /content/RVRT/testsets/processing_temp --num_workers 1 --save_result

        # Se duplicamos frames no final, removemos as duplicatas processadas da pasta de resultados
        if end_idx > total_frames:
            for i in range(total_frames - start_idx, chunk_size):
                dup_out = final_results_dir / f'frame_{start_idx + i:08d}.png'
                if dup_out.exists():
                    dup_out.unlink()

        # Liberação explícita de memória do cache da GPU
        gc.collect()
        torch.cuda.empty_cache()
else:
    print('[ERRO] Nenhum frame foi encontrado para processamento. Verifique se o upload na etapa 2 foi concluído com sucesso.')
    print('\nDiagnóstico de diretórios:')
    print(f'Opção 1 ({path_option_1}): Existe? {path_option_1.exists()} | Qtd frames: {len(list(path_option_1.glob("*.png"))) if path_option_1.exists() else 0}')
    print(f'Opção 2 ({path_option_2}): Existe? {path_option_2.exists()} | Qtd frames: {len(list(path_option_2.glob("*.png"))) if path_option_2.exists() else 0}')

print('\nProcessamento finalizado!')

In [ ]:
import glob
from pathlib import Path

results_path = 'results/001_RVRT_videosr_bi_REDS_30frames/processing_temp/*.png'
processed_frames = glob.glob(results_path)
num_frames = len(processed_frames)

print(f'Quantidade de frames encontrados na pasta de resultados: {num_frames}')
if num_frames == 1184:
    print('Sucesso! Todos os 1184 frames foram gerados e salvos.')
else:
    print(f'Atenção: Foram encontrados apenas {num_frames} de 1184 frames esperados. O processamento ainda pode estar em andamento ou foi interrompido.')

In [ ]:
import subprocess

# Executa o comando 'ps aux' para listar os processos ativos
ps_output = subprocess.check_output(['ps', 'aux']).decode('utf-8')

# Filtra por processos do python que estejam rodando a nossa inferência
active_runs = [line for line in ps_output.split('\n') if 'main_test_rvrt.py' in line]

print('=== Verificação de Processo de Inferência ===\n')
if active_runs:
    print('STATUS: O processo de inferência do RVRT está ATIVO no sistema!')
    for run in active_runs:
        print(f'Detalhes: {run}')
else:
    print('STATUS: O processo de inferência NÃO está ativo no momento (Parado, Interrompido ou Concluído).')

```markdown
## 7. Compilação dos Resultados em um Vídeo MP4 de Alta Qualidade
Esta etapa lê todos os frames de super-resolução processados e reconstrói o seu vídeo original com máxima nitidez.
```

In [ ]:
from google.colab import files

output_video_name = 'video_restaurado_qualidade_maxima.mp4'

print('Compilando os frames processados em vídeo...')
# Compila as imagens sequenciais no codec x264 a 25 fps de forma contínua
!ffmpeg -y -framerate 25 -pattern_type glob -i 'results/001_RVRT_videosr_bi_REDS_30frames/processing_temp/*.png' -c:v libx264 -pix_fmt yuv420p -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" {output_video_name}

print(f'\nProcesso concluído! Vídeo disponível: {output_video_name}')
print('Iniciando o download do seu vídeo final de alta resolução...')
files.download(output_video_name)